# CorrDiff - Fase 7 - Dependência Temporal e Lags

Análise de autocorrelação, persistência de eventos e associações preditor(t-k) → radar(t).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
OUT=Path("../analysis_outputs/07_lags")
summary=json.loads((OUT/"analysis_summary.json").read_text())
summary


## 1. Cobertura dos pares temporais

In [ ]:
cov=pd.read_parquet(OUT/"lag_pair_coverage.parquet")
display(cov)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(cov.lag_hours,cov.pair_availability_ratio,marker="o")
ax.set_xlabel("Lag (h)")
ax.set_ylabel("Disponibilidade de pares exatos")
ax.set_ylim(0,1)
plt.tight_layout(); plt.show()


## 2. Autocorrelação do radar

In [ ]:
rad=pd.read_parquet(OUT/"radar_lag_autocorrelation.parquet")
for metric in ["max_dbz","positive_pixel_fraction","event_pixel_fraction_ge_40"]:
    t=rad[rad.metric.eq(metric)].sort_values("lag_hours")
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot(t.lag_hours,t.spearman_rho_raw,marker="o",label="bruto")
    ax.plot(t.lag_hours,t.spearman_rho_month_hour_adjusted,marker="o",label="ajustado mês×hora")
    ax.set_xlabel("Lag (h)"); ax.set_ylabel("Spearman"); ax.set_title(metric); ax.legend()
    plt.tight_layout(); plt.show()


## 3. Dependência dos eventos

In [ ]:
ev=pd.read_parquet(OUT/"event_lag_dependence.parquet")
for event_id in ["ge_30","ge_40","ge_45"]:
    t=ev[ev.event_id.eq(event_id)].sort_values("lag_hours")
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot(t.lag_hours,t.p_event_t_given_event_t_minus_lag,marker="o",label="P(evento_t | evento_t-k)")
    ax.plot(t.lag_hours,t.p_event_t_given_no_event_t_minus_lag,marker="o",label="P(evento_t | não-evento_t-k)")
    ax.set_xlabel("Lag (h)"); ax.set_ylabel("Probabilidade"); ax.set_title(event_id); ax.legend()
    plt.tight_layout(); plt.show()


## 4. Preditores lagados dos eventos

In [ ]:
pe=pd.read_parquet(OUT/"predictor_event_lag_associations.parquet")
event_id="ge_40"
for lag in sorted(pe.lag_hours.unique()):
    t=pe[(pe.event_id.eq(event_id))&(pe.lag_hours.eq(lag))].copy()
    t["abs_corr"]=t.corr_month_hour_adjusted.abs()
    display(t.nlargest(10,"abs_corr")[["lag_hours","predictor","source","point_biserial_r_raw","corr_month_hour_adjusted"]])


## 5. Preditores lagados do máximo em dBZ

In [ ]:
pr=pd.read_parquet(OUT/"predictor_radar_lag_associations.parquet")
for lag in sorted(pr.lag_hours.unique()):
    t=pr[(pr.target_metric.eq("max_dbz"))&(pr.lag_hours.eq(lag))].copy()
    t["abs_corr"]=t.spearman_rho_month_hour_adjusted.abs()
    display(t.nlargest(10,"abs_corr")[["lag_hours","predictor","source","spearman_rho_raw","spearman_rho_month_hour_adjusted"]])


## Interpretação
Compare sempre métricas brutas e ajustadas. Se uma associação desaparecer após remover mês×hora, ela provavelmente é dominada pela climatologia. Associação residual em lags curtos indica informação temporal adicional, mas não causalidade.